## Load data from hf

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from datasets import load_dataset

In [ ]:
os.getenv("HF_TOKEN")

In [ ]:
df = load_dataset("viamr-project/reasoning-frame-hint", token=os.getenv("HF_TOKEN"))
df

In [ ]:
sample = df["train"][0]

# Hint

In [ ]:
from services.amr_hint.amr_hint import AMRHint

amr = """
(w / want-01
    :ARG0 (b / boy)
    :ARG1 (g / go-01
              :ARG0 b))
""".strip()

hint_gen = AMRHint()
hints = hint_gen.get_hints_json(amr)
print(hints)

In [ ]:
from services.amr_hint.prompt_builder import build_prompt

print(build_prompt(sample["sentence"], sample["amr"]))

In [ ]:
partial = hint_gen.get_hints_partial(amr, percentage=0.3)
print(partial)

In [ ]:
import json

# JSON (string) - phù hợp để đưa thẳng vào model
print(hint_gen.get_hints_json(amr, indent=2))

# Hoặc lấy dict Python (structured object)
structured = hint_gen.get_hints_structured(amr)
structured

# vLLM

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
import os
from vllm import SamplingParams

vllm_config = {
    "model": "Qwen/Qwen3.6-35B-A3B-FP8",
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.85,
    "max_model_len": 4096,
    
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repetition_penalty": 1.0,
    "max_tokens": 1024,
}

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sampling_params = SamplingParams(
    temperature=vllm_config["temperature"],
    top_p=vllm_config["top_p"],
    top_k=vllm_config["top_k"],
    min_p=vllm_config["min_p"],
    presence_penalty=vllm_config["presence_penalty"],
    repetition_penalty=vllm_config["repetition_penalty"],
    max_tokens=vllm_config["max_tokens"],
)

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")


try:
    vllm_model = LLM(
        model=vllm_config["model"],
        tensor_parallel_size=vllm_config["tensor_parallel_size"],
        gpu_memory_utilization=vllm_config["gpu_memory_utilization"],
        max_model_len=vllm_config["max_model_len"],
        hf_token=os.getenv("HF_TOKEN")
    )
    print(f"✓ vLLM model loaded successfully: {vllm_config["model"]}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from vllm import LLM

prompt = "AMR trong NLP là gì?"

outputs = vllm_model.generate(prompt, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
print(generated_text)

In [ ]:
sample = {"amr":"(n / need-01\n   :ARG0 (h / he)\n   :ARG1 (c / conduct-01\n            :ARG0 h\n            :ARG1 (m / meet-03\n                     :ARG0 (p / person\n                              :name (n2 / name\n                                        :op1 \"Palin\"))\n                     :ARG1 (p2 / publication\n                               :name (n3 / name\n                                         :op1 \"NBC\"))\n                     :mod (s / summit\n                             :mod (b / beer)))))","sentence":"He needs to conduct a beer summit between Palin and NBC.","model_respose":["Let me analyze this carefully.\n\nThe sentence is: \"He needs to conduct a beer summit between Palin and NBC.\"\n\nKey elements:\n1. \"He\" - a person (ARG0 of need-01)\n2. \"needs\" - main predicate `need-01`\n3. \"to conduct a beer summit\" - ARG1 of need-01, with `conduct-01` as sub-predicate\n4. \"beer summit\" - ARG1 of conduct-01\n5. \"between Palin and NBC\" - this is a `meet-03` event, with Palin and NBC as the parties\n\nLet me trace through:\n\nRoot: `need-01`\n- ARG0: `person` (he)\n- ARG1: `conduct-01`\n  - ARG0: `person` (same \"he\")\n  - ARG1: `meet-03` (the \"beer summit\" is a meeting)\n    - ARG0: `person` \"Palin\"\n    - ARG1: `organization` \"NBC\"\n\nNow, \"beer summit\" modifies the meet-03 event - it describes the type of meeting (a summit involving beer). I'll treat \"beer\" as a modifier on the meeting domain.\n\nLet me also consider that \"conduct\" might take the summit as a whole event rather than just the participants. So `conduct-01`'s ARG1 is the `meet-03` event.\n\nFor names: \"Palin\" and \"NBC\" are named entities. \"Palin\" is a person, \"NBC\" is an organization.\n\nLet me build the AMR:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (m / meet-03\n                     :ARG0 (p2 / person :name (n2 / name :op1 \"Palin\"))\n                     :ARG1 (o / organization :name (n3 / name :op1 \"NBC\"))\n                     :mod (b / beer))))\n```\n\nWait, let me reconsider. \"Beer summit\" is a named event concept - it's a specific type of meeting. Maybe \"summit\" is the head noun and \"beer\" is a modifier. But in AMR, \"summit\" as a meeting would likely map to `meet-03` or similar.\n\nActually, looking at the hints more carefully:\n- The frames given are: need-01, conduct-01, meet-03\n- meet-03: \"get together (with)\", ARG0 and ARG1 are the two parties\n\nSo \"a beer summit between Palin and NBC\" maps to a `meet-03` event with Palin and NBC as the two parties, and \"beer\" as a modifier.\n\nLet me produce the AMR:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (m / meet-03\n                     :ARG0 (p2 / person :name (n2 / name :op1 \"Palin\"))\n                     :ARG1 (o / organization :name (n3 / name :op1 \"NBC\"))\n                     :mod (b / beer))))\n```\n\nThis seems right. The \"he\" is the same person conducting the meeting, so I reused the person variable `p` for conduct-01's ARG0.<think>\nThe sentence structure is:\n- \"He needs to conduct a beer summit between Palin and NBC.\"\n- Main predicate: need-01, with \"He\" as ARG0 (the needer) and the infinitive clause as ARG1 (thing needed).\n- The infinitive \"to conduct a beer summit between Palin and NBC\" uses conduct-01.\n  - conduct-01's ARG0 is the same \"he\" (reusing the person variable).\n  - conduct-01's ARG1 is \"a beer summit between Palin and NBC\".\n- \"summit\" here is a meeting event → meet-03.\n  - meet-03's ARG0 = Palin (person)\n  - meet-03's ARG1 = NBC (organization)\n  - \"beer\" is a modifier on the meet-03 event (type of summit).\n- Names are represented with :name and :op1.\n</think>\n\n<amr>\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (m / meet-03\n                     :ARG0 (p2 / person :name (n2 / name :op1 \"Palin\"))\n                     :ARG1 (o / organization :name (n3 / name :op1 \"NBC\"))\n                     :mod (b / beer))))\n</amr>"]}


sample["model_respose"][0]

In [ ]:
print(sample["model_respose"][0])

In [ ]:
import json

with open("data/systhetic_data.jsonl", "r") as f:
    data = [json.loads(line) for line in f if line.strip()]

In [ ]:
data[0]

In [ ]:
import json
from pathlib import Path

output_path = Path("data/systhetic_data.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

output_path

## Đánh giá độ chính xác với smatchpp

In [ ]:
import json
import re
from smatchpp import Smatchpp

# ── helpers ────────────────────────────────────────────────────────────────────

def extract_amr_from_response(model_response: list[str] | str) -> str | None:
    """
    Extract AMR string from <amr>...</amr> tags in model_response.
    model_response can be a list of strings or a plain string.
    Returns the first match found, or None.
    """
    text = "\n".join(model_response) if isinstance(model_response, list) else model_response
    match = re.search(r"<amr>(.*?)</amr>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def score_amr_pair(gold: str, pred: str, scorer: Smatchpp) -> dict:
    """
    Score a single (gold, pred) AMR pair.
    Returns {'F1': float, 'Precision': float, 'Recall': float}
    or {'F1': 0.0, 'Precision': 0.0, 'Recall': 0.0} on parse errors.
    """
    try:
        result = scorer.score_pair(gold, pred)
        return result["main"]          # {'F1': ..., 'Precision': ..., 'Recall': ...}
    except Exception as e:
        return {"F1": 0.0, "Precision": 0.0, "Recall": 0.0, "error": str(e)}


# ── main ───────────────────────────────────────────────────────────────────────

def evaluate(data_path: str = "data/systhetic_data.jsonl") -> None:
    with open(data_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    scorer = Smatchpp()

    results = []
    skipped = 0

    for i, item in enumerate(data):
        gold_amr       = item.get("amr", "").strip()
        sentence       = item.get("sentence", "")
        model_response = item.get("model_respose", item.get("model_response", ""))  # typo-tolerant

        pred_amr = extract_amr_from_response(model_response)

        if not gold_amr or not pred_amr:
            print(f"[{i}] SKIP — missing {'gold' if not gold_amr else 'pred'} AMR | sentence: {sentence!r}")
            skipped += 1
            continue

        scores = score_amr_pair(gold_amr, pred_amr, scorer)

        results.append({
            "index":     i,
            "sentence":  sentence,
            "gold_amr":  gold_amr,
            "pred_amr":  pred_amr,
            **scores,
        })

        print(
            f"[{i:>4}] F1={scores['F1']:5.1f}  "
            f"P={scores['Precision']:5.1f}  "
            f"R={scores['Recall']:5.1f}  | {sentence[:70]}"
        )

    # ── aggregate ──────────────────────────────────────────────────────────────
    if results:
        avg_f1   = sum(r["F1"]        for r in results) / len(results)
        avg_prec = sum(r["Precision"] for r in results) / len(results)
        avg_rec  = sum(r["Recall"]    for r in results) / len(results)

        print("\n" + "=" * 70)
        print(f"  Evaluated : {len(results)} samples  (skipped: {skipped})")
        print(f"  Avg F1    : {avg_f1:.2f}")
        print(f"  Avg Prec  : {avg_prec:.2f}")
        print(f"  Avg Recall: {avg_rec:.2f}")
        print("=" * 70)
    else:
        print("No valid pairs to evaluate.")

    # ── save detailed results ──────────────────────────────────────────────────
    out_path = "amr_eval_results.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            # convert numpy floats → plain Python floats for JSON
            row = {k: (float(v) if hasattr(v, "item") else v) for k, v in r.items()}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"\nDetailed results saved to: {out_path}")


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else "data/systhetic_data.jsonl"
    evaluate(path)

In [ ]:
import json
import re
from smatchpp import Smatchpp

# ── helpers ────────────────────────────────────────────────────────────────────

def extract_amr_from_response(model_response: list[str] | str) -> str | None:
    """
    Extract AMR string from <amr>...</amr> tags in model_response.
    model_response can be a list of strings or a plain string.
    Returns the first match found, or None.
    """
    text = "\n".join(model_response) if isinstance(model_response, list) else model_response
    match = re.search(r"<amr>(.*?)</amr>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def score_amr_pair(gold: str, pred: str, scorer: Smatchpp) -> dict:
    """
    Score a single (gold, pred) AMR pair.
    Returns {'F1': float, 'Precision': float, 'Recall': float}
    or {'F1': 0.0, 'Precision': 0.0, 'Recall': 0.0} on parse errors.
    """
    try:
        result = scorer.score_pair(gold, pred)
        return result["main"]          # {'F1': ..., 'Precision': ..., 'Recall': ...}
    except Exception as e:
        return {"F1": 0.0, "Precision": 0.0, "Recall": 0.0, "error": str(e)}


# ── main ───────────────────────────────────────────────────────────────────────

def evaluate(data_path: str = "data/systhetic_data.jsonl") -> None:
    with open(data_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    scorer = Smatchpp()

    results = []
    skipped = 0

    for i, item in enumerate(data):
        gold_amr       = item.get("amr", "").strip()
        sentence       = item.get("sentence", "")
        model_response = item.get("model_respose", item.get("model_response", ""))  # typo-tolerant

        pred_amr = extract_amr_from_response(model_response)

        if not gold_amr or not pred_amr:
            print(f"[{i}] SKIP — missing {'gold' if not gold_amr else 'pred'} AMR | sentence: {sentence!r}")
            skipped += 1
            continue

        scores = score_amr_pair(gold_amr, pred_amr, scorer)

        results.append({
            "index":     i,
            "sentence":  sentence,
            "gold_amr":  gold_amr,
            "pred_amr":  pred_amr,
            **scores,
        })

        print(
            f"[{i:>4}] F1={scores['F1']:5.1f}  "
            f"P={scores['Precision']:5.1f}  "
            f"R={scores['Recall']:5.1f}  | {sentence[:70]}"
        )

    # ── aggregate ──────────────────────────────────────────────────────────────
    if results:
        avg_f1   = sum(r["F1"]        for r in results) / len(results)
        avg_prec = sum(r["Precision"] for r in results) / len(results)
        avg_rec  = sum(r["Recall"]    for r in results) / len(results)

        print("\n" + "=" * 70)
        print(f"  Evaluated : {len(results)} samples  (skipped: {skipped})")
        print(f"  Avg F1    : {avg_f1:.2f}")
        print(f"  Avg Prec  : {avg_prec:.2f}")
        print(f"  Avg Recall: {avg_rec:.2f}")
        print("=" * 70)
    else:
        print("No valid pairs to evaluate.")

    # ── save detailed results ──────────────────────────────────────────────────
    out_path = "amr_eval_results.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            # convert numpy floats → plain Python floats for JSON
            row = {k: (float(v) if hasattr(v, "item") else v) for k, v in r.items()}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"\nDetailed results saved to: {out_path}")


if __name__ == "__main__":
    evaluate("data/systhetic_data.jsonl")

In [2]:
sample_1 ={"amr":"(n / need-01\n   :ARG0 (h / he)\n   :ARG1 (c / conduct-01\n            :ARG0 h\n            :ARG1 (m / meet-03\n                     :ARG0 (p / person\n                              :name (n2 / name\n                                        :op1 \"Palin\"))\n                     :ARG1 (p2 / publication\n                               :name (n3 / name\n                                         :op1 \"NBC\"))\n                     :mod (s / summit\n                             :mod (b / beer)))))","sentence":"He needs to conduct a beer summit between Palin and NBC.","model_respose":["Let’s break this down step by step.\n\nThe sentence is:  \n**\"He needs to conduct a beer summit between Palin and NBC.\"**\n\n### Step 1 – Identify the main predicate  \nThe main verb is \"needs\" → **need-01**.  \n- ARG0: the one who needs → \"He\" → a person.  \n- ARG1: the thing needed → \"to conduct a beer summit between Palin and NBC\" → a **conduct-01** event.\n\n### Step 2 – Handle the embedded clause  \n\"conduct a beer summit between Palin and NBC\"  \n- \"conduct-01\" is the event.  \n- ARG0 of conduct: the conductor → \"He\" (same as the needer).  \n- ARG1 of conduct: the thing led → \"beer summit\".  \n- \"beer summit\" is a **summit** with a modifier \"beer\".  \n- \"between Palin and NBC\" indicates a **meet-03** relation (get together) between the two parties.  \n  - Palin: a person.  \n  - NBC: a publication (a network, channel, etc.).  \n\n### Step 3 – Structure the AMR  \n- Top: need-01  \n  - ARG0: (p / person :name \"He\")  \n  - ARG1: (c / conduct-01  \n      - ARG0: same as needer  \n      - ARG1: (s / summit  \n          - :mod (b / beer)  \n          - :ARG1-of (m / meet-03  \n              - ARG0 (p2 / person :name \"Palin\")  \n              - ARG1 (n / publication :name \"NBC\")))  \n  - Wait, \"between Palin and NBC\" is a prepositional phrase indicating a meeting between them. So the summit is the event that is led, and the meeting is the event that happens between them.\n\nActually, better: \"between Palin and NBC\" is a **meet-03** event whose participants are Palin and NBC. That meet-03 is the event that is part of the \"beer summit\". In AMR, \"between\" often becomes a **:domain** or **:ARG1-of** relation.\n\nLet's think:  \n\"conduct a beer summit between Palin and NBC\"  \n- The summit is a **summit**.  \n- It has a modifier \"beer\".  \n- It is \"between Palin and NBC\" → the summit is the event of meeting between them. So we can use **:ARG1-of (m / meet-03 ...)** on the summit.\n\n### Step 4 – Write the AMR  \n```\n(n / need-01\n   :ARG0 (h / person :name \"He\")\n   :ARG1 (c / conduct-01\n            :ARG0 h\n            :ARG1 (s / summit\n                     :mod (b / beer)\n                     :ARG1-of (m / meet-03\n                                :ARG0 (p / person :name \"Palin\")\n                                :ARG1 (n / publication :name \"NBC\")))))\n```\n\n### Step 5 – Check consistency  \n- \"He\" appears as ARG0 of need and ARG0 of conduct.  \n- \"Beer summit\" is a single concept: summit with mod beer.  \n- \"Between Palin and NBC\" uses meet-03 with participants Palin and NBC.  \n- \"NBC\" is a publication, which fits the given hint.\n\nAll hints are respected.\n\nNow produce final output.\nThe sentence is \"He needs to conduct a beer summit between Palin and NBC.\"\n\nThe main predicate is \"needs\" → **need-01**, with ARG0 = \"He\" (a person) and ARG1 = the event of conducting a beer summit.\n\nThe embedded event is \"conduct a beer summit between Palin and NBC\" → **conduct-01**, with ARG0 = the same person (He), and ARG1 = \"beer summit\".\n\nThe \"beer summit\" is a **summit** with a modifier **beer**. The phrase \"between Palin and NBC\" indicates a meeting between those two entities, so I use **meet-03** as a relation on the summit: the summit is the ARG1-of the meeting (i.e., the meeting event is the one that happens between them).\n\nPalin is a person, NBC is a publication. The names are given as strings.\n\nThe structure is:\n- need-01\n  - ARG0: person named \"He\"\n  - ARG1: conduct-01\n    - ARG0: same person\n    - ARG1: summit\n      - mod: beer\n      - ARG1-of: meet-03\n        - ARG0: person named \"Palin\"\n        - ARG1: publication named \"NBC\"\n</think>\n\n<amr>\n(n / need-01 :ARG0 (h / he) :ARG1 (c / conduct-01 :ARG0 h :ARG1 (s / summit :topic (b / beer) :ARG1-of (m / meet-03 :ARG0 (p / person :name (n2 / name :op1 \"Palin\")) :ARG1 (p2 / publication :name (n3 / name :op1 \"NBC\"))))))\n</amr>","Let me break down the sentence: \"He needs to conduct a beer summit between Palin and NBC.\"\n\nFirst, the main predicate is \"need-01\" with ARG0 (needer) = \"he\" and ARG1 (thing needed) = \"conduct a beer summit between Palin and NBC\".\n\nNow, \"conduct-01\" has ARG0 = \"he\" (the conductor) and ARG1 = \"beer summit\" (the thing led).\n\n\"Beer summit\" is a compound noun - \"summit\" is the head, \"beer\" is a modifier (likely \"beer summit\" meaning a summit of beer or a beer summit event).\n\n\"Between Palin and NBC\" - this is a prepositional phrase indicating the parties involved. \"meet-03\" with ARG0 = \"Palin\" and ARG1 = \"NBC\" - this is the location or context of the summit.\n\nActually, \"between Palin and NBC\" could be interpreted as the parties involved in the summit. Let me think about this more carefully.\n\n\"Beer summit\" - this is a specific event. \"summit\" is a concept, \"beer\" is a modifier. The \"between\" phrase could indicate the parties involved in the summit.\n\nLet me reconsider: \"conduct a beer summit between Palin and NBC\" - the \"between\" phrase might be a location or context. Alternatively, it could mean the summit is between these two parties.\n\nLooking at the hints: we have \"conduct-01\" with ARG0 and ARG1. The ARG1 is \"beer summit\" which is a concept. The \"between\" phrase likely needs to be represented somehow.\n\nFor \"between Palin and NBC\", I think this is best represented as a conjunction or as a location/context. One approach: use \"meet-03\" with ARG0 and ARG1 to indicate the parties involved.\n\nActually, let me think about this differently. \"Between Palin and NBC\" could be a location or a manner. In AMR, prepositions are often represented with :location or :domain.\n\nLet me structure this:\n\n1. Root: need-01\n   - ARG0: he (person)\n   - ARG1: conduct-01\n     - ARG0: he (same person)\n     - ARG1: beer-summit (summit with modifier beer)\n       - :location or :domain: (meet-03)\n         - ARG0: person (Palin)\n         - ARG1: publication (NBC) or person (NBC)\n\nWait, \"NBC\" is a name, so it should be a named entity. Let me think about this more carefully.\n\n\"Beer summit\" - this is a specific event. In AMR, we might represent it as a concept \"summit\" with a modifier \"beer\".\n\n\"Between Palin and NBC\" - this is a prepositional phrase indicating the parties involved. \"Between\" here might indicate the location or the context of the summit.\n\nLet me look at this as: \"He needs to conduct a beer summit, and this summit is between Palin and NBC.\"\n\nFor the \"between\" part, I think we could use :location with \"meet-03\" or use a conjunction.\n\nActually, in AMR, \"between\" as a preposition can be represented as a relation. One approach is to use :domain or :location.\n\nLet me try:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / summit\n                     :mod (b / beer)\n                     :location (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (n2 / publication :name \"NBC\"))))\n```\n\nHmm, but this doesn't quite capture the relationship between \"beer summit\" and \"between Palin and NBC\" well enough.\n\nLet me reconsider. The \"between\" phrase might be a separate concept or it could be integrated.\n\nActually, I think the most natural reading is: He needs to conduct a beer summit, and the summit is between Palin and NBC. So the \"between\" phrase modifies the summit.\n\nIn AMR, we could represent this as:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / summit\n                     :mod (b / beer)\n                     :location (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (n2 / publication :name \"NBC\"))))\n```\n\nWait, but :location with meet-03 might not be the best way to represent \"between\". Let me think about this differently.\n\n\"Between Palin and NBC\" - this could be a conjunction or a prepositional phrase. In AMR, we often use :domain or :location for such prepositions.\n\nActually, I think a cleaner representation would be to use a conjunction or just treat \"between\" as a relation. Let me think about what's most standard.\n\nIn AMR, prepositions are often represented using a special relation. For \"between\", I think we could use something like:\n\n```\n:location (b / between\n            :op1 (p / person :name \"Palin\")\n            :op2 (n / publication :name \"NBC\"))\n```\n\nOr we could use a more standard approach. Let me look at this more carefully.\n\nActually, I think the simplest and most accurate representation is:\n\nFor \"between Palin and NBC\", we can use a conjunction or just use the meet-03 frame as a location.\n\nLet me go with:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / summit\n                     :mod (b / beer)\n                     :location (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (n2 / publication :name \"NBC\"))))\n```\n\nHmm, but :location with meet-03 might not be the best way. Let me think about whether there's a standard way to represent \"between\" in AMR.\n\nActually, in AMR, \"between\" as a preposition can be represented as a relation. One common approach is to use :domain or :location with a conjunction or specific relation.\n\nLet me try another approach. What if \"between Palin and NBC\" is treated as a location or as a modifier of the summit?\n\nI think the most natural reading is: \"He needs to conduct a beer summit, and the summit is between Palin and NBC.\"\n\nIn AMR, we could represent this as:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / summit\n                     :mod (b / beer)\n                     :location (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (n2 / publication :name \"NBC\"))))\n```\n\nThis seems reasonable. The \"between\" part is represented as a meet-03 event where Palin and NBC meet (the parties involved in the summit). The summit is the location/context of this meeting.\n\nActually, I want to reconsider. The hints mention \"meet-03\" with ARG0 and ARG1, which makes sense for \"between\" in the context of a summit.\n\nLet me also consider: \"beer summit\" - is \"beer\" a modifier of \"summit\" or is it a separate concept? \"Beer summit\" is a compound noun where \"beer\" modifies \"summit\" to mean a summit of beer. So :mod (b / beer) seems right.\n\nFor \"NBC\", since it's a publication, we should use the concept \"publication\" as given in the hints.\n\nLet me finalize my approach. I think this is a reasonable AMR:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / summit\n                     :mod (b / beer)\n                     :location (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (n2 / publication :name \"NBC\"))))\n```\n\nWait, but I need to double-check: does \"need-01\" take ARG0 and ARG1 as per the hints? The hints say:\n- need-01: ARG0 = needer, ARG1 = thing needed\n- conduct-01: ARG0 = conductor, ARG1 = thing led\n- meet-03: ARG0 = one party, ARG1 = other party\n\nYes, this matches. The ARG0 of need-01 is \"he\", and the ARG1 is the conduct-01 event. The ARG0 of conduct-01 is also \"he\". The ARG0 of meet-03 is \"Palin\" and ARG1 is \"NBC\".\n\nI think this is a good representation. Let me output it.\n</think>\n\n<amr>\n(n / need-01 :ARG0 (h / he) :ARG1 (c / conduct-01 :ARG0 h :ARG1 (s / summit :mod (b / beer) :location (m / meet-03 :ARG0 (p / person :name (n2 / name :op1 \"Palin\")) :ARG1 (p2 / publication :name (n3 / name :op1 \"NBC\"))))))\n</amr>","Let me break down the sentence: \"He needs to conduct a beer summit between Palin and NBC.\"\n\nThe main predicate is \"need-01\" with ARG0 being \"he\" and ARG1 being the thing needed, which is \"conduct-01\".\n\nFor \"conduct-01\", the ARG0 is the conductor, which is \"he\" (the same as the needer), and ARG1 is \"a beer summit\".\n\nThe \"beer summit\" is a name for a meeting or gathering. The meeting involves \"Palin\" and \"NBC\" as participants.\n\nLet me think about the structure:\n\n- need-01\n  - ARG0: he (person)\n  - ARG1: conduct-01\n    - ARG0: he (same person)\n    - ARG1: beer-summit (name)\n      - :ARG0-of (or :ARG1-of?) for the participants? Actually, \"between Palin and NBC\" means Palin and NBC are participants in the summit.\n\nFor the meeting, we could use \"meet-03\" with ARG0 and ARG1 being Palin and NBC. But \"beer summit\" is a named event. Let me think...\n\nActually, \"beer summit\" is a named event, and \"between Palin and NBC\" indicates the participants. So:\n\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (or :ARG1-of?) for the participants? Actually, \"between\" in \"between Palin and NBC\" means they are the participants.\n\nLet me reconsider. In AMR, \"meet-03\" is used for meetings. So:\n\n- meet-03\n  - ARG0: person (Palin)\n  - ARG1: person (NBC)\n  - :location beer-summit\n\nBut \"beer-summit\" is the name of the event. So:\n\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (or :ARG1-of?) - hmm.\n\nActually, \"between\" indicates the participants. In AMR, we might use a different approach. Let me think about this more carefully.\n\nThe sentence structure:\n- He needs (need-01)\n  - ARG0: he (person)\n  - ARG1: conduct-01\n    - ARG0: he (same person)\n    - ARG1: beer-summit\n      - :name \"beer-summit\"\n      - :ARG0-of (or :ARG1-of?) - participants\n\nFor the participants, we can use \"person\" for both Palin and NBC, and relate them to the meeting.\n\nLet me try a different approach. The \"beer summit\" is a meeting event. The participants are Palin and NBC.\n\nSo:\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (or :ARG1-of?) - participants\n\nActually, in AMR, when you have a meeting between two people, you might use:\n- meet-03\n  - ARG0: person (Palin)\n  - ARG1: person (NBC)\n  - :location beer-summit\n\nBut \"beer-summit\" is the name, not the location. Let me think...\n\nMaybe:\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (or :ARG1-of?) - participants\n\nI think the most natural reading is that the meeting has participants Palin and NBC, and the meeting is called \"beer-summit\".\n\nLet me look at the hints again. The hints give us:\n- need-01 with ARG0 and ARG1\n- conduct-01 with ARG0 and ARG1\n- meet-03 with ARG0 and ARG1\n- person concept\n- publication concept\n\nThe hints don't mention the participants explicitly, but \"between Palin and NBC\" indicates they are participants.\n\nLet me construct the AMR:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nWait, I'm overcomplicating this. Let me think about what \"between\" means here. \"Between Palin and NBC\" means they are the participants of the meeting. In AMR, we could use :ARG0-of or :ARG1-of for the participants.\n\nActually, in AMR, when you have a meeting between two people, you might use:\n- meet-03\n  - ARG0: person (Palin)\n  - ARG1: person (NBC)\n  - :location beer-summit\n\nBut \"beer-summit\" is the name of the event, not the location. Let me think about this differently.\n\nMaybe:\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (m / meet-03\n               :ARG0 (p / person :name \"Palin\")\n               :ARG1 (p2 / person :name \"NBC\"))\n\nThis says the meeting (between Palin and NBC) is the ARG0 of beer-summit. That doesn't quite work either.\n\nLet me reconsider. Perhaps the simplest approach:\n\nThe \"beer summit\" is a meeting. The participants are Palin and NBC. The meeting is called \"beer-summit\".\n\nIn AMR, we could represent this as:\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (m / meet-03\n               :ARG0 (p / person :name \"Palin\")\n               :ARG1 (p2 / person :name \"NBC\"))\n\nWhere :ARG0-of means the meeting is the ARG0 of beer-summit. That seems backwards.\n\nLet me try another approach. Maybe the meeting is the thing being conducted, and the participants are the ones who are met.\n\nActually, I think the cleanest way is:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nHmm, but :ARG0-of on beer-summit means beer-summit is the ARG0 of meet-03, which is wrong.\n\nLet me try:\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nWait, I'm not sure :ARG0-of is the right relation. Let me think about what \"between Palin and NBC\" means in AMR.\n\nIn AMR, \"between\" can be represented as a relation. The participants are Palin and NBC. They are the ARG0 and ARG1 of meet-03.\n\nMaybe the beer-summit is the ARG0 of meet-03, and the participants are the ARG0 and ARG1? No, that doesn't make sense.\n\nLet me look at this from a different angle. The sentence says \"He needs to conduct a beer summit between Palin and NBC.\"\n\nSo the main event is conduct-01. The thing being conducted is a beer-summit. The beer-summit involves Palin and NBC as participants.\n\nIn AMR, for a meeting between two people, we typically use meet-03 with ARG0 and ARG1.\n\nSo:\n- beer-summit\n  - :name \"beer-summit\"\n  - :ARG0-of (m / meet-03\n               :ARG0 (p / person :name \"Palin\")\n               :ARG1 (p2 / person :name \"NBC\"))\n\nThis says the meeting between Palin and NBC is the ARG0 of beer-summit. That's backwards - the meeting IS beer-summit.\n\nActually, I think the correct reading is that the meeting (between Palin and NBC) is the beer-summit. So beer-summit is the name of the meeting, and the meeting has participants Palin and NBC.\n\nLet me try:\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nHmm, I'm going in circles. Let me try a different representation.\n\nPerhaps the simplest AMR is:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nWait, I think the issue is that :ARG0-of is not the right relation. Let me think about what the actual AMR convention is for \"between\".\n\nIn AMR, \"between X and Y\" can be represented as:\n- X and Y are participants\n- The relation between them is \"between\"\n\nFor meetings, we typically use meet-03 with ARG0 and ARG1.\n\nLet me try a cleaner approach:\n\nThe beer-summit is a meeting. The participants are Palin and NBC.\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nActually, I think the most natural reading is that the meeting is beer-summit. So beer-summit is the name, and it's the ARG0 of meet-03, where the participants are Palin and NBC.\n\nLet me try:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nI think this is the most natural AMR representation. The beer-summit is the name of the meeting, and the meeting has participants Palin and NBC.\n\nActually, I realize I should double-check: in AMR, is :ARG0-of a standard relation? Let me think... In AMR, :ARG0-of means \"the ARG0 of this concept is the thing that comes after it\". So if we have:\n\n(m / meet-03\n   :ARG0 (p / person :name \"Palin\")\n   :ARG1 (p2 / person :name \"NBC\"))\n\nThis means the meeting has ARG0 Palin and ARG1 NBC. So :ARG0-of on meet-03 would mean the meeting is the ARG0 of something, which doesn't make sense.\n\nHmm, I think I'm overthinking this. Let me just go with a reasonable representation.\n\nActually, let me reconsider the structure. Maybe the beer-summit is the main concept, and the meeting is a sub-event:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nI think this is the most natural reading. The beer-summit is the name, and it's the ARG0 of the meet-03 event, meaning the meeting is the beer-summit. The participants are Palin and NBC.\n\nWait, I just realized that :ARG0-of on meet-03 might not be standard. Let me think about what AMR convention says about this.\n\nIn AMR, for relations like \"between\", we often use :ARG0-of or :ARG1-of. For example, \"the meeting between X and Y\" could be represented as:\n\n(m / meet-03\n   :ARG0 (p / person :name \"Palin\")\n   :ARG1 (p2 / person :name \"NBC\"))\n\nOr we could use a different relation. Let me just go with what makes sense.\n\nI think the most natural AMR for this sentence is:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (s / beer-summit\n                     :name \"beer-summit\"\n                     :ARG0-of (m / meet-03\n                                 :ARG0 (p2 / person :name \"Palin\")\n                                 :ARG1 (p3 / person :name \"NBC\")))))\n```\n\nThis says:\n- He needs (need-01)\n  - ARG0: he (person)\n  - ARG1: conduct-01\n    - ARG0: he (person)\n    - ARG1: beer-summit (name: beer-summit)\n      - beer-summit is the ARG0 of meet-03 (the meeting is the beer-summit)\n        - ARG0: person (Palin)\n        - ARG1: person (NBC)\n\nThis seems reasonable. The \"between\" relation is captured by the participants being the ARG0 and ARG1 of meet-03.\n\nLet me finalize this.\n</think>\n\n<amr>\n(n / need-01 :ARG0 (h / he) :ARG1 (c / conduct-01 :ARG0 h :ARG1 (s / summit :topic (b / beer) :ARG0-of (m / meet-03 :ARG1 (p / person :name (n2 / name :op1 \"Palin\")) :ARG1 (p2 / publication :name (n3 / name :op1 \"NBC\"))))))\n</amr>"]}
sample_2 ={"amr":"(s / stick-around-03\n   :ARG0 (i / i)\n   :time (u / until\n            :op1 (e / end-01)))","sentence":"I will stick around until the end.","model_respose":["Let me analyze this sentence step by step.\n\nThe sentence is: \"I will stick around until the end.\"\n\nFrom the knowledge provided, I know:\n- The root frame is \"stick-around-03\" which means \"stick around: continue to be in a location, hang out\"\n- The frame has ARG0 as \"sticker around, hanger out\"\n- There's also \"end-01\" which means \"cause to stop, coming to a close\"\n\nThe sentence structure:\n- \"I\" is the subject - this is ARG0 of stick-around-03\n- \"will stick around\" - this is the main predicate\n- \"until the end\" - this is a temporal modifier\n\nFor \"until the end\":\n- \"end\" is a concept, likely \"end-01\" as per the knowledge\n- \"until\" indicates a temporal relation\n\nLet me think about how to structure this:\n\nThe main predicate is stick-around-03 with ARG0 = \"I\"\nThe temporal modifier \"until the end\" needs to be attached to the main event.\n\nIn AMR, \"until\" can be represented as a temporal relation. \"end\" is likely the concept for \"end-01\" (the event of ending).\n\nSo the structure would be:\n- (s / stick-around-03\n    :ARG0 (i / i)\n    :time (e / end-01))\n\nBut wait, \"until\" is a temporal relation. In AMR, we typically use :time for temporal expressions like \"until\".\n\nActually, let me reconsider. \"until the end\" means \"until the event of ending occurs\" or \"until the end of something\". The \"end\" here refers to the end of the situation or event.\n\nLet me think about this more carefully. \"I will stick around until the end\" means:\n- I will continue to be in the location (or the situation) until the end of some event happens.\n\nIn AMR, \"until\" is typically represented as a :time relation with a concept like \"end-01\" or as a temporal relation. Actually, for \"until\", we often use :time with a :op1 or similar structure.\n\nActually, let me look at how AMR typically handles \"until\". In AMR, \"until\" is often represented as:\n- (t / temporal-quantity :quant 1 :unit (e / end-01))\n\nOr more commonly, \"until\" can be represented as a :time relation with a concept like \"end-01\".\n\nLet me think about this differently. The sentence means \"I will continue to be present until the end of the situation\". The \"end\" here refers to the end of the situation or the end of the event.\n\nI think the simplest representation would be:\n- (s / stick-around-03\n    :ARG0 (i / i)\n    :time (e / end-01))\n\nBut wait, \"until\" might need a more explicit temporal relation. In AMR, \"until\" is often represented with :time and a :op1 or similar.\n\nActually, looking at standard AMR practices, \"until\" can be represented as:\n- (u / until :op1 (e / end-01))\n\nOr more commonly, it's represented as a :time relation with a concept like \"end-01\".\n\nLet me go with a cleaner representation. The sentence is \"I will stick around until the end.\"\n\nI'll use:\n- stick-around-03 as the root\n- ARG0 = i (the speaker)\n- A temporal modifier indicating \"until the end\"\n\nFor \"until\", in AMR we can use a temporal relation with the concept \"end-01\".\n\nLet me try:\n(s / stick-around-03\n    :ARG0 (i / i)\n    :time (e / end-01))\n\nThis seems reasonable. The \"end\" represents the event of ending, and \"until\" indicates this event is the endpoint of the duration.\n\nActually, let me reconsider. In some AMR annotations, \"until\" is represented as a :time relation with a :op1. But I think using a simple :time with end-01 is fine.\n\nLet me also consider: is \"end-01\" the right frame? The knowledge says end-01 has meaning \"cause to stop, coming to a close\" with ARG0, ARG1, ARG2. But in this sentence, \"the end\" is just a concept, not necessarily an event with an agent or thing ending. So I think using end-01 as a concept is appropriate here.\n\nI'll go with this structure.\n</think>\n\n<amr>\n(s / stick-around-03 :ARG0 (i / i) :time (e / end-01))\n</amr>","Let me analyze this sentence step by step.\n\nThe sentence is: \"I will stick around until the end.\"\n\nFirst, let's identify the main predicate. The core action is \"stick around\" which maps to the frame `stick-around-03`. This means \"continue to be in a location, hang out.\"\n\nThe subject \"I\" is the ARG0 of `stick-around-03`.\n\nThe phrase \"until the end\" indicates a temporal constraint. The \"end\" here likely refers to the end of the event or the event itself. Looking at the frames, `end-01` is listed, which means \"cause to stop, coming to a close.\" The ARG1 of `end-01` would be the thing ending. In this context, \"the end\" refers to the end of the situation or the event.\n\nThe word \"until\" suggests a temporal relation. In AMR, \"until\" can be represented as a temporal relation with `:time` or as a conjunction. Actually, \"until\" in this context means \"continue until the end\" — it's a continuation up to a point in time. This is often represented as a `:duration` or using the `:time` relation with an `:op2` or similar.\n\nLet me think about how to structure this. The main predicate is `stick-around-03`. The ARG0 is \"I\". The ARG1 (thing being stuck around) is not explicitly stated — it could be implicit (the location is implied or generic). The ARG2 (subevent) might not be used.\n\nFor \"until the end\", I need to represent this as a temporal constraint. In AMR, \"until\" can be represented as a conjunction or as a `:time` relation with an `:op2` indicating the endpoint. Actually, a common approach is to use `:time` with a temporal relation like `:op2` or `:duration` with a `:quant` or `:time` pointing to an event.\n\nLet me think more carefully. The phrase \"stick around until the end\" means \"continue to be in a location up to the end of the event.\" The \"end\" here is the end of the situation. So the `end-01` frame has ARG1 as the thing ending. In this case, \"the end\" is the end of something (the event, the situation, etc.), so it's the ARG1 of `end-01`.\n\nFor \"until\", I think the best representation is to use a `:time` relation with an `:op2` or to use `:duration` with a `:quant` or `:time` pointing to the end event. Actually, in AMR, \"until\" can be represented as a conjunction or as a temporal relation. Let me use a standard approach: `:time` with an `:op2` that is the endpoint.\n\nWait, let me reconsider. In AMR, \"until\" can be represented using the `:time` relation with an `:op2` that is the endpoint, or using a special relation like `:duration` with `:quant` of \"until\" or something. Actually, I think the most standard approach is:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation\n                     :ARG1-of (e2 / end-01))))\n```\n\nHmm, that's getting complicated. Let me think again.\n\nThe sentence is: \"I will stick around until the end.\"\n\nThe main predicate is `stick-around-03`. The subject \"I\" is ARG0. The phrase \"until the end\" is a temporal modifier.\n\nFor \"until the end\", I think the simplest representation is to use a `:time` relation with an `:op2` that represents the endpoint. But actually, in AMR, \"until\" is often represented as a conjunction or using a special relation.\n\nLet me look at this differently. The phrase \"stick around until the end\" can be interpreted as \"continue to be in a location up to the end of the event.\" So the end event is the endpoint of the sticking around.\n\nOne common AMR pattern for \"until\" is:\n\n```\n:time (e / end-01\n         :ARG1 (s / situation))\n```\n\nBut that doesn't quite work because \"the end\" is not an event in the sense of ending something. The end is the endpoint of the sticking around.\n\nActually, I think a cleaner approach is:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation\n                     :ARG1-of (e2 / end-01))))\n```\n\nNo, that's circular. Let me try a different approach.\n\nMaybe the simplest representation is:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (e2 / end-01)))\n```\n\nWait, that's not right either. \"The end\" is a noun, not an event. The end of what? The sentence doesn't specify what is ending — it's generic.\n\nLet me think about this more simply. In AMR, \"until\" can be represented as:\n\n```\n:time (e / end-01\n         :ARG1 (x / thing))\n```\n\nBut that's not quite right either because \"the end\" is not an event that ends something. The end is a concept.\n\nActually, I think the best representation is:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation\n                     :ARG1-of (e2 / end-01))))\n```\n\nHmm, this is getting too complicated. Let me simplify.\n\nActually, I think the most straightforward representation for \"until the end\" is to use a `:duration` or `:time` relation that points to the end event. Let me look at this from a different angle.\n\nIn AMR, temporal expressions like \"until\" are often represented using the `:time` relation with a special relation like `:op2` or using the `:duration` relation. But I think the most common approach is:\n\n```\n:time (e / end-01\n         :ARG1 (x / thing))\n```\n\nBut this doesn't capture \"until\" well. Let me think about what AMR guidelines say.\n\nIn AMR, \"until\" can be represented as:\n\n```\n:time (e / end-01\n         :ARG1 (s / situation))\n```\n\nThis means \"until the end of the situation\" which is close enough. The \"until\" is captured by the `:time` relation pointing to the end event.\n\nActually, I think the most natural reading is that \"the end\" is the endpoint of the sticking around. So the sticking around continues up to the end of some event. The event is implicit.\n\nLet me go with a simpler representation:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (x / thing)))\n```\n\nBut I'm not sure if `:ARG1` should be used for \"the end\" as the ARG1 of `end-01`. The frame says `end-01` has ARG1 = \"Thing ending\". So \"the end\" as a concept is not the thing ending — it's the endpoint. \n\nLet me reconsider. Perhaps:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation)))\n```\n\nThis reads as \"until the end of the situation\" which is a reasonable interpretation.\n\nActually, I think the cleanest representation is:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation\n                     :ARG1-of (e2 / end-01))))\n```\n\nNo, that's circular. Let me just use a simpler approach.\n\nLet me look at this from the perspective of what the AMR should capture. The sentence is \"I will stick around until the end.\" The main predicate is `stick-around-03` with ARG0 = \"I\". The phrase \"until the end\" is a temporal modifier.\n\nFor \"until\", I think the most standard AMR representation uses `:time` with a special relation. Let me just use:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (x / thing)))\n```\n\nThis is clean and captures the meaning that the sticking around continues until the end of some event. The \"thing\" is generic — it could be any event.\n\nActually, I think a more precise representation would be:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation\n                     :ARG1-of (e2 / end-01))))\n```\n\nWait, but this creates a cycle. The situation is ARG1 of end-01, and end-01's ARG1 is situation. That's fine — it means the situation ends.\n\nBut actually, \"the end\" in this context refers to the endpoint of the sticking around, not the end of the situation. The \"end\" is a noun, not an event.\n\nLet me just go with a simpler, cleaner representation. I'll use:\n\n```\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (x / thing)))\n```\n\nThis reads as \"until the end of a thing\" which is close enough. The \"thing\" is generic and captures the implicit event.\n\nActually, I think I'm overcomplicating this. Let me just go with a straightforward representation. The main predicate is `stick-around-03`, ARG0 is \"I\", and there's a temporal modifier \"until the end\".\n\nFor \"until the end\", I'll use:\n\n```\n:time (e / end-01\n         :ARG1 (s / situation))\n```\n\nThis means \"until the end of a situation\" which captures the meaning well.\n\nLet me finalize this. The sentence is \"I will stick around until the end.\" The main predicate is `stick-around-03`. ARG0 is \"I\". The temporal modifier \"until the end\" uses `end-01` as the endpoint, with `:ARG1` being a generic situation.\n\nFinal AMR:\n</think>\n\n<amr>\n(s / stick-around-03 :ARG0 (i / i) :duration (e / end-01))\n</amr>","Let me analyze the sentence \"I will stick around until the end.\"\n\nFirst, let's identify the main predicate. The core meaning is about continuing to be present in a location (stick around). The frame \"stick-around-03\" has ARG0 as the one doing the sticking around. Here, \"I\" is the ARG0.\n\nThe phrase \"until the end\" indicates a temporal constraint. \"end\" is the concept for the endpoint. The relation \"until\" indicates a temporal boundary.\n\nLooking at the frames provided:\n- stick-around-03: ARG0 = sticker around, hanger out\n- end-01: ARG0 = Agent/cause of ending, ARG1 = Thing ending, ARG2 = Subevent or thing that Arg1 ends with\n\nFor \"stick around until the end\":\n- The main event is stick-around-03 with ARG0 = \"I\"\n- The \"until\" indicates a temporal relation - it's the endpoint\n- \"the end\" refers to the end of something (likely the whole situation)\n\nIn AMR, \"until\" is typically represented with a temporal relation. The \"end\" is the endpoint. The ARG1 of end-01 would be the thing that ends, which could be the situation being discussed (implied).\n\nLet me think about how to structure this:\n\nThe main event is stick-around-03.\n- ARG0: i (the speaker)\n- ARG1: (implicit - the location being stuck around)\n- The \"until\" indicates a temporal boundary - this could be represented with a :time relation or with end-01\n\nActually, in AMR, \"until\" can be represented as a temporal relation. The phrase \"until the end\" means \"up to the point when something ends.\"\n\nLet me think about this more carefully. The sentence is \"I will stick around until the end.\"\n\nThe main predicate is stick-around-03. The ARG0 is \"I\". The ARG1 (the thing being stuck around) is implicit - it's the location or situation.\n\nThe \"until\" indicates a temporal constraint. In AMR, this is often represented with a :time relation or with a special relation like :duration or :time.\n\nFor \"until the end\", I think the best representation is to use end-01 as a temporal modifier. The \"end\" is the endpoint, and \"until\" indicates the boundary.\n\nOne common way to handle \"until\" in AMR is to use the concept \"end\" with a temporal relation. For example:\n\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation)))\n\nBut wait, \"end\" here refers to the end of something. In context, \"the end\" likely refers to the end of the situation or the event. Without more context, we can represent it as a generic \"end\" concept.\n\nActually, let me reconsider. The sentence is \"I will stick around until the end.\" The \"end\" is a noun referring to the endpoint. In AMR, we can represent it as:\n\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation)))\n\nBut we don't know what \"the end\" refers to - it could be the end of the conversation, the end of the situation, etc. We can use a generic concept.\n\nAlternatively, we could use a simpler representation. The \"until\" can be represented as a temporal relation with a polarity or as a separate concept.\n\nLet me look at this differently. The sentence structure is:\n- Main: stick-around-03 (ARG0 = I)\n- Temporal: until the end\n\nIn AMR, \"until\" is often represented as:\n- A :time relation with the endpoint\n- Or as a special relation like :duration\n\nFor \"until the end\", I think the best approach is:\n\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation)))\n\nThis captures that the sticking around continues until the end of the situation.\n\nActually, I think there's another way to handle this. The \"until\" could be represented with a :time relation that has a temporal concept. But I think using end-01 as the endpoint is cleaner.\n\nLet me also consider if there's a more standard way. In AMR, \"until\" is often represented as a temporal relation. The \"end\" is the endpoint.\n\nI'll go with this structure:\n\n(s / stick-around-03\n   :ARG0 (i / i)\n   :time (e / end-01\n            :ARG1 (s2 / situation)))\n\nThis seems reasonable. The \"end\" refers to the endpoint of the situation, and \"until\" indicates the temporal boundary.\n\nWait, I need to double-check the frame information. The frame for end-01 says:\n- ARG0: Agent/cause of ending\n- ARG1: Thing ending\n- ARG2: Subevent or thing that Arg1 ends with\n\nIn \"the end\" as a noun phrase, \"the end\" is the endpoint. So ARG1 of end-01 would be the thing that ends, which could be the situation being discussed.\n\nLet me finalize this. I think this is a solid representation.\n</think>\n\n<amr>\n(s / stick-around-03 :ARG0 (i / i) :time (e / end-01))\n</amr>"]}

In [3]:
sample_2["model_respose"]

['Let me analyze this sentence step by step.\n\nThe sentence is: "I will stick around until the end."\n\nFrom the knowledge provided, I know:\n- The root frame is "stick-around-03" which means "stick around: continue to be in a location, hang out"\n- The frame has ARG0 as "sticker around, hanger out"\n- There\'s also "end-01" which means "cause to stop, coming to a close"\n\nThe sentence structure:\n- "I" is the subject - this is ARG0 of stick-around-03\n- "will stick around" - this is the main predicate\n- "until the end" - this is a temporal modifier\n\nFor "until the end":\n- "end" is a concept, likely "end-01" as per the knowledge\n- "until" indicates a temporal relation\n\nLet me think about how to structure this:\n\nThe main predicate is stick-around-03 with ARG0 = "I"\nThe temporal modifier "until the end" needs to be attached to the main event.\n\nIn AMR, "until" can be represented as a temporal relation. "end" is likely the concept for "end-01" (the event of ending).\n\nSo the